# 🛡️ SpamShield AI — Clasificación de SMS Spam con Deep Learning

## Objetivos

1. Cargar y explorar el dataset SMS Spam Collection de UCI.
2. Realizar un análisis exploratorio de datos (EDA).
3. Limpiar y preprocesar los textos.
4. Construir un modelo BiLSTM con TensorFlow/Keras.
5. Entrenar y evaluar el modelo.
6. Guardar el modelo y realizar predicciones.

## 1. Importación de librerías

In [ ]:
import json
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from tensorflow.keras import callbacks, layers, models

print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 2. Obtención y descripción del dataset

**Fuente:** [UCI Machine Learning Repository — SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection)

**Problema:** Clasificación binaria de mensajes SMS como spam o legítimos (ham).

**Descripción:** El dataset contiene mensajes SMS reales etiquetados manualmente. Es ampliamente utilizado en investigación de detección de spam.

In [ ]:
# Rutas
ROOT_DIR = Path.cwd()
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
RESULTS_DIR = ROOT_DIR / "results"
MODELS_DIR = ROOT_DIR / "models"

# Buscar el archivo del dataset
dataset_path = RAW_DIR / "SMSSpamCollection"
print(f"Dataset encontrado: {dataset_path.exists()}")
print(f"Ruta: {dataset_path}")

In [ ]:
# Cargar el dataset
df = pd.read_csv(
    dataset_path,
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="latin-1",
)

print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nPrimeras filas:")
df.head(10)

In [ ]:
# Información del dataset
print("=== INFORMACIÓN DEL DATASET ===")
print(f"Número de registros: {len(df)}")
print(f"Número de columnas: {df.shape[1]}")
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nValores nulos:")
print(df.isnull().sum())
print(f"\nRegistros duplicados: {df.duplicated().sum()}")

In [ ]:
# Distribución de clases
print("=== DISTRIBUCIÓN DE CLASES ===")
class_counts = df["label"].value_counts()
class_pcts = df["label"].value_counts(normalize=True) * 100
for label in class_counts.index:
    print(f"{label:5s}: {class_counts[label]:5d} registros ({class_pcts[label]:.2f}%)")

### Descripción de los campos

| Campo | Tipo | Descripción |
|---|---|---|
| `label` | string | Clase del mensaje: `ham` (legítimo) o `spam` (no deseado) |
| `message` | string | Texto completo del mensaje SMS |

**Variable objetivo:** `label` (clasificación binaria)

**Limitaciones y sesgos:**
- Desbalance de clases (~13% spam, ~87% ham).
- Los mensajes provienen de un período y región específicos.
- Puede contener información personal (números de teléfono, nombres).
- El dataset está en inglés únicamente.

## 3. Análisis exploratorio de datos (EDA)

In [ ]:
# Calcular longitudes
df["length_chars"] = df["message"].str.len()
df["length_words"] = df["message"].str.split().str.len()

print("=== ESTADÍSTICAS DE LONGITUD ===")
print(df[["length_chars", "length_words"]].describe())

In [ ]:
# Gráfico 1: Distribución de clases
fig, ax = plt.subplots(figsize=(8, 5))
class_counts.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"])
ax.set_title("Distribución de clases en el dataset", fontsize=14)
ax.set_xlabel("Clase", fontsize=12)
ax.set_ylabel("Número de mensajes", fontsize=12)
ax.set_xticklabels(["Ham (legítimo)", "Spam"], rotation=0)
for i, (count, pct) in enumerate(zip(class_counts.values, class_pcts.values)):
    ax.text(i, count + 50, f"{count}\n({pct:.1f}%)", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=150)
plt.show()

In [ ]:
# Gráfico 2: Histograma de longitud de mensajes
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df["length_chars"], bins=50, color="#3498db", edgecolor="white", alpha=0.8)
ax.set_title("Distribución de la longitud de mensajes (caracteres)", fontsize=14)
ax.set_xlabel("Longitud en caracteres", fontsize=12)
ax.set_ylabel("Frecuencia", fontsize=12)
ax.axvline(df["length_chars"].mean(), color="red", linestyle="--", label=f"Media: {df['length_chars'].mean():.0f}")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "message_lengths.png", dpi=150)
plt.show()

In [ ]:
# Gráfico 3: Comparación de longitud spam vs ham
fig, ax = plt.subplots(figsize=(10, 6))
for label, color in [("ham", "#2ecc71"), ("spam", "#e74c3c")]:
    subset = df[df["label"] == label]["length_chars"]
    ax.hist(subset, bins=40, alpha=0.6, label=f"{label} (media={subset.mean():.0f})", color=color)
ax.set_title("Comparación de longitud: Spam vs Ham", fontsize=14)
ax.set_xlabel("Longitud en caracteres", fontsize=12)
ax.set_ylabel("Frecuencia", fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Ejemplos reales de cada categoría
print("=== EJEMPLOS DE MENSAJES HAM ===")
for msg in df[df["label"] == "ham"]["message"].head(3):
    print(f"  • {msg[:100]}..." if len(msg) > 100 else f"  • {msg}")

print("\n=== EJEMPLOS DE MENSAJES SPAM ===")
for msg in df[df["label"] == "spam"]["message"].head(3):
    print(f"  • {msg[:100]}..." if len(msg) > 100 else f"  • {msg}")

### Interpretación del EDA

1. **Desbalance de clases:** El ~87% de los mensajes son ham y solo ~13% spam. Esto significa que un modelo que siempre predice "ham" tendría ~87% de accuracy, pero sería inútil. Por ello es crucial monitorear precision, recall y F1-score.

2. **Longitud de mensajes:** Los mensajes spam tienden a ser más largos que los ham. Esta característica es útil para el modelo.

3. **Distribución asimétrica:** La mayoría de los mensajes son cortos (<100 caracteres), con una cola larga de mensajes extensos.

## 4. Limpieza y preprocesamiento

In [ ]:
# Limpieza del dataset
print(f"Registros antes de limpieza: {len(df)}")

# Eliminar nulos
df = df.dropna(subset=["label", "message"])

# Convertir a string
df["message"] = df["message"].astype(str)

# Normalizar etiquetas
df["label"] = df["label"].str.strip().str.lower()

# Eliminar mensajes vacíos
df = df[df["message"].str.strip() != ""]

# Normalizar espacios
df["message"] = df["message"].str.replace(r"\s+", " ", regex=True).str.strip()

# Eliminar duplicados
df = df.drop_duplicates(subset=["message"]).reset_index(drop=True)

print(f"Registros después de limpieza: {len(df)}")
print(f"\nDistribución final:")
print(df["label"].value_counts())

In [ ]:
# Codificar etiquetas
label_map = {"ham": 0, "spam": 1}
df["label_encoded"] = df["label"].map(label_map)

# Ejemplos antes y después
print("=== EJEMPLOS DE PREPROCESAMIENTO ===")
for i in range(3):
    print(f"\nOriginal:  {df['message'].iloc[i][:80]}")
    print(f"Etiqueta:  {df['label'].iloc[i]} → {df['label_encoded'].iloc[i]}")

### Justificación de las transformaciones

| Transformación | Justificación |
|---|---|
| Eliminar nulos | No aportan información al modelo |
| Eliminar duplicados | Evita sobre-representación de patrones |
| Normalizar espacios | Reduce ruido sin perder contenido semántico |
| `lower_and_strip_punctuation` (en TextVectorization) | Normalización estándar; el modelo aprende patrones sin distinguir mayúsculas |
| **Conservar números y enlaces** | Son indicadores clave de spam (ej: "$", "http", "click") |

**Nota sobre data leakage:** La capa `TextVectorization` se adapta ÚNICAMENTE con datos de entrenamiento.

## 5. División del dataset (70/15/15 estratificada)

In [ ]:
RANDOM_SEED = 42

X = df["message"].values
y = df["label_encoded"].values

# Primera división: train (70%) vs temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)

# Segunda división: val (15%) vs test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"Train: {len(X_train)} ejemplos ({(len(X_train)/len(X)*100):.1f}%)")
print(f"Val:   {len(X_val)} ejemplos ({(len(X_val)/len(X)*100):.1f}%)")
print(f"Test:  {len(X_test)} ejemplos ({(len(X_test)/len(X)*100):.1f}%)")

# Verificar estratificación
for name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    spam_pct = (y_split == 1).mean() * 100
    print(f"{name}: {spam_pct:.1f}% spam")

## 6. Construcción del modelo

**Justificación de BiLSTM:**
- Los mensajes spam tienen patrones que dependen del contexto completo.
- BiLSTM captura información tanto de izquierda a derecha como de derecha a izquierda.
- Es más efectivo que LSTM simple para textos cortos donde el orden importa.

In [ ]:
# Hiperparámetros
VOCAB_SIZE = 10000
SEQUENCE_LENGTH = 200
EMBEDDING_DIM = 128
LSTM_UNITS = 64
DROPOUT_RATE_1 = 0.3
DROPOUT_RATE_2 = 0.2
BATCH_SIZE = 64
EPOCHS = 20
PATIENCE = 3

# Fijar semillas
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Capa de vectorización
vectorize_layer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation",
)

# Construir modelo
model = models.Sequential([
    vectorize_layer,
    layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True),
    layers.Bidirectional(layers.LSTM(LSTM_UNITS, return_sequences=False)),
    layers.Dropout(DROPOUT_RATE_1),
    layers.Dense(32, activation="relu"),
    layers.Dropout(DROPOUT_RATE_2),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# Adaptar vectorización SOLO con datos de entrenamiento
model.layers[0].adapt(X_train)
print(f"Vocabulario: {model.layers[0].vocabulary_size()} tokens")
model.summary()

## 7. Entrenamiento

In [ ]:
# Callbacks
early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1,
)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = MODELS_DIR / "spamshield_lstm.keras"

model_checkpoint = callbacks.ModelCheckpoint(
    filepath=str(checkpoint_path),
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
)

# Entrenar
print("Iniciando entrenamiento...")
start_time = time.time()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping, model_checkpoint],
    verbose=1,
)

elapsed = time.time() - start_time
print(f"\nEntrenamiento completado en {elapsed:.1f}s")
print(f"Épocas ejecutadas: {len(history.history['loss'])}")

## 8. Gráficas de aprendizaje

In [ ]:
# Gráfica de pérdida
fig, ax = plt.subplots(figsize=(10, 6))
epochs_range = range(1, len(history.history["loss"]) + 1)
ax.plot(epochs_range, history.history["loss"], "b-", label="Pérdida entrenamiento", linewidth=2)
ax.plot(epochs_range, history.history["val_loss"], "r-", label="Pérdida validación", linewidth=2)
ax.set_title("Pérdida de entrenamiento vs validación", fontsize=14)
ax.set_xlabel("Época", fontsize=12)
ax.set_ylabel("Pérdida (binary crossentropy)", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_loss.png", dpi=150)
plt.show()

In [ ]:
# Gráfica de accuracy
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs_range, history.history["accuracy"], "b-", label="Accuracy entrenamiento", linewidth=2)
ax.plot(epochs_range, history.history["val_accuracy"], "r-", label="Accuracy validación", linewidth=2)
ax.set_title("Accuracy de entrenamiento vs validación", fontsize=14)
ax.set_xlabel("Época", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_accuracy.png", dpi=150)
plt.show()

### Interpretación de las gráficas

- **Sobreajuste:** Si la pérdida de entrenamiento sigue bajando pero la de validación sube, el modelo está memorizando los datos.
- **Subajuste:** Si ambas curvas permanecen altas, el modelo no está aprendiendo suficiente.
- **Convergencia ideal:** Ambas curvas bajan juntas y se estabilizan.

## 9. Evaluación con conjunto de prueba

In [ ]:
# Cargar el mejor modelo guardado
best_model = tf.keras.models.load_model(str(checkpoint_path))

# Predicciones
y_prob = best_model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

# Métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=== MÉTRICAS GLOBALES ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred, target_names=["Ham (0)", "Spam (1)"]))

## 10. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"True Negatives (ham correcto):  {tn}")
print(f"False Positives (ham → spam):    {fp}")
print(f"False Negatives (spam → ham):    {fn}")
print(f"True Positives (spam correcto):  {tp}")

# Gráfico
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Ham (0)", "Spam (1)"],
            yticklabels=["Ham (0)", "Spam (1)"], ax=ax)
ax.set_title("Matriz de confusión — Conjunto de prueba", fontsize=14)
ax.set_xlabel("Etiqueta predicha", fontsize=12)
ax.set_ylabel("Etiqueta real", fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150)
plt.show()

### Interpretación de errores

- **Falsos Positives (ham → spam):** Mensajes legítimos clasificados como spam. Pueden ser mensajes con lenguaje similar al spam (ofertas, urgencia).
- **Falsos Negativos (spam → ham):** Mensajes spam que el modelo no detectó. Son los más peligrosos porque el usuario recibiría el spam.

## 11. Predicciones con ejemplos del conjunto de prueba

In [ ]:
# Seleccionar 12 ejemplos del conjunto de prueba
test_df = pd.DataFrame({"message": X_test, "label": y_test})
sample_df = test_df.sample(n=12, random_state=RANDOM_SEED).reset_index(drop=True)

results = []
for _, row in sample_df.iterrows():
    prob = float(best_model.predict(np.array([row["message"]]), verbose=0).flatten()[0])
    pred = 1 if prob >= 0.5 else 0
    results.append({
        "Mensaje": row["message"][:60] + "..." if len(row["message"]) > 60 else row["message"],
        "Etiqueta real": "spam" if row["label"] == 1 else "ham",
        "Etiqueta predicha": "spam" if pred == 1 else "ham",
        "Probabilidad spam": f"{prob:.4f}",
        "Resultado": "✅ Correcta" if pred == row["label"] else "❌ Incorrecta",
    })

results_df = pd.DataFrame(results)
results_df

## 12. Guardado del modelo

In [ ]:
# El modelo ya fue guardado por ModelCheckpoint durante el entrenamiento
print(f"Modelo guardado en: {checkpoint_path}")
print(f"Existe: {checkpoint_path.exists()}")
print(f"Tamaño: {checkpoint_path.stat().st_size / 1024:.1f} KB")

# Guardar configuración
config = {
    "vocab_size": VOCAB_SIZE,
    "sequence_length": SEQUENCE_LENGTH,
    "embedding_dim": EMBEDDING_DIM,
    "lstm_units": LSTM_UNITS,
    "epochs_trained": len(history.history["loss"]),
    "training_time_seconds": round(elapsed, 2),
    "best_val_loss": round(min(history.history["val_loss"]), 4),
    "best_val_accuracy": round(max(history.history["val_accuracy"]), 4),
    "best_epoch": int(np.argmin(history.history["val_loss"])) + 1,
}

config_path = MODELS_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Configuración guardada: {config_path}")

## 13. Prueba de carga y predicción

In [ ]:
# Cargar el modelo guardado y verificar que funciona
loaded_model = tf.keras.models.load_model(str(checkpoint_path))

test_messages = [
    "Congratulations! You've won a $1000 gift card. Click here to claim.",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account has been compromised. Call now!",
    "Thanks for the birthday wishes!",
]

print("=== PREDICCIONES CON MODELO CARGADO ===")
for msg in test_messages:
    prob = float(loaded_model.predict(tf.constant([msg], dtype=tf.string), verbose=0).flatten()[0])
    label = "SPAM" if prob >= 0.5 else "HAM"
    print(f"\nMensaje: {msg[:60]}...")
    print(f"  → {label} (probabilidad: {prob:.4f})")

## 14. Conclusiones

1. El modelo BiLSTM logra clasificar mensajes SMS con alta precisión.
2. La arquitectura con TextVectorization integrada permite predecir directamente sobre texto crudo.
3. El uso de EarlyStopping y ModelCheckpoint evita el sobreajuste.
4. El modelo guardado puede cargarse y utilizarse sin reentrenamiento.

## 15. Limitaciones y posibles mejoras

**Limitaciones:**
- Solo funciona con mensajes en inglés.
- No analiza imágenes ni enlaces.
- Puede tener falsos positivos en mensajes legítimos con lenguaje similar al spam.

**Posibles mejoras:**
- Usar embeddings pre-entrenados (Word2Vec, GloVe).
- Probar arquitecturas Transformer (BERT).
- Aplicar técnicas de balanceo de clases (SMOTE, class weights).
- Incluir características adicionales (presencia de números, enlaces, mayúsculas).